# 06 – Teams

Shared credit pools (teams). Team has its own balance; members draw
from the pool. Per-member spend caps available.

Useful for organisations, projects, or departments sharing credits.

In [1]:
from datetime import datetime, timedelta
from ducto.interface.postgres import PostgresStore
from ducto.manager import CreditManager
from ducto.engine import PricingEngine
from ducto.metrics import UsageMetrics, ToolCall
from ducto.interface.models import (
    PricingConfigData, PricingConfigV2, PlanDefinition,
    CreditMetadata,
)
from shared import start_postgres_store, cleanup

store, pgdata = start_postgres_store()
import uuid
print("✔ PostgresStore ready.")

Initialising Postgres cluster …


✔ PostgresStore ready.


### Create team with initial balance

In [2]:
team = store.create_team(name="Engineering", initial_balance=100_000)
print(f"  Team: {team.name}  (id={team.team_id})")

  Team: Engineering  (id=bd6633b8-276d-4ebd-84a3-74acec1928eb)


### Add members

In [3]:
members = [str(uuid.uuid4()) for _ in range(3)]
for uid in members:
    store.add_credits(uid, 0, type="adjustment")  # user must exist in user_credits
    store.add_team_member(team.team_id, uid, role="member")
    print(f"  Added {uid[:8]}…")

bal = store.get_team_balance(team.team_id)
print(f"  Team balance: {bal.balance}  Members: {bal.member_count}")

  Added 9ef58c51…
  Added 2b4e6d43…
  Added b8b87788…
  Team balance: 100000  Members: 3


### Deduct from team pool

In [4]:
res = store.deduct_team(team.team_id, members[0], 5_000)
print(f"  balance_after={res.team_balance_after}  error={res.error}")

bal2 = store.get_team_balance(team.team_id)
assert bal2.balance == 95_000

  balance_after=95000  error=None


### Exceed team balance (rejected)

In [5]:
res2 = store.deduct_team(team.team_id, members[1], 999_999)
print(f"  error={res2.error}  balance={res2.team_balance_after}")
assert res2.error == "insufficient_team_balance" 

  error=insufficient_team_balance  balance=0


In [6]:
cleanup(pgdata)

Cleaned up.
